# Study overview and revised conclusions

This repository contains the corrected experiments prepared for the Scientific Reports revision. Read this notebook first. The two questions are point-prediction performance and local uncertainty, evaluated separately.

**Mode:** executed analysis of the committed corrected results. Expensive model refitting is available through Notebook 11 and `scripts/reproduce.py`. Replaying saved results is not presented as fresh model training.

In [1]:
from pathlib import Path
import sys, json, itertools
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.revision-repository').exists())
sys.path.insert(0, str(ROOT / 'analysis'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from notebook_support import *
from analysis import PROTEINS, TARGET, FIXED, global_quantile, local_intervals
pd.set_option('display.max_columns', 14)
pd.set_option('display.max_rows', 30)
pd.set_option('display.precision', 5)


## What changed

The public ADMET block had 1,678 shifted compound associations. The corrected primary cohort contains **12,584** compounds; the exact-identity sensitivity cohort contains **12,193**. These are re-associated existing ADMET outputs, not new API predictions. All experiments are retrospective; the marker candidates and original configurations had already been developed on this collection.

In [2]:
repair = read_json(PROCESSED/'repair_summary.json')
display(pd.Series({k: repair[k] for k in ['premerge_master_n','published_wrong_ADMET_association_n','primary_n','strict_n']}, name='Count').to_frame())
assert repair['primary_n'] == 12584 and repair['strict_n'] == 12193

,Count
premerge_master_n,12652
published_wrong_ADMET_association_n,1678
primary_n,12584
strict_n,12193


## Point prediction

The ADME configuration now has lower RMSE than Plain in every primary partition. These are specified configurations with different frozen hyperparameters, not a matched causal feature-addition experiment. BBB models require a matched subgroup comparison (Notebook 06).

In [3]:
g = table('global_models/metrics.csv')
full = g[g.model.ne('BBB pass')]
display(full.groupby('model')[['rmse','mae','r2','coverage_0.9']].agg(['mean','std']))
paired = table('strict_global_comparison.csv')
assert (paired.rmse_ADME < paired.rmse_Plain).all()
display(paired.groupby('cohort')[['rmse_ADME','rmse_Plain']].mean())

rmse               mae                r2          coverage_0.9  \
             mean      std     mean      std     mean      std         mean   
model                                                                         
ADME      0.49304  0.00671  0.34202  0.00261  0.52014  0.01411      0.90632   
Baseline  0.52501  0.01009  0.36666  0.00347  0.45603  0.01307      0.90250   
PCA       0.52442  0.01208  0.36476  0.00612  0.45729  0.01534      0.90025   
Plain     0.51217  0.01610  0.35453  0.00751  0.48257  0.01273      0.90269   

                   
              std  
model              
ADME      0.00835  
Baseline  0.01575  
PCA       0.00659  
Plain     0.01321

,rmse_ADME,rmse_Plain
cohort,,
primary,0.49304,0.51217
strict_exact_identity,0.49284,0.51677


## Uncertainty beyond the stronger predictor

Biological localization remains useful on the stronger ADME predictor. Width is interpreted together with coverage and interval score. The latter penalizes both wide intervals and missed observations; smaller is better. No uniform local 90% guarantee is claimed.

In [4]:
strong = table('strong_predictor_calibration/metrics.csv')
display(strong.groupby(['model','method'])[['coverage','mean_width','mean_interval_score']].mean())

coverage  mean_width  mean_interval_score
model method                                           
ADME  global   0.90632     1.52475              2.33449
      joint    0.89620     1.42761              2.25713
Plain global   0.90269     1.56708              2.42028
      joint    0.89343     1.47448              2.34695

## Reading order

01 identity repair → 02 splits → 03 global models → 04 calibration → 05 SHAP and permutations → 06 matched BBB → 07 biological coordinates and normalization → 08 robustness controls → 09 maps and thresholds → 10 historical/related-repository diagnostics → 11 reproduction and reviewer map → 12 figure gallery.

**Two predictor definitions:** the global Plain model uses master-table MW, logP and molecular fingerprints. The original confidence-analysis configuration uses ADMET MW, logP and fingerprints with fixed 800-tree settings. Its results are therefore labeled separately.

**Unresolved provenance:** historical Butina cutoff, full source-study metadata and exact upstream ADMET predictor training sets/version were not recovered. Their absence is documented rather than filled with assumed values.